In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nfl.db")
team_games = pd.read_sql("SELECT * FROM team_games", conn)

print(team_games.shape)

(6600, 98)


In [ ]:
# only rows with real outcomes and real stats can be used for training/testing
# Please Note taht - 2026 games have NULL scores, and 2025 has incomplete weekly stats
modeling_data = team_games[team_games["season"] <= 2024].copy()

# get the unique list of games to  split by game_id by keeping both rows of any game together
unique_games = modeling_data[["game_id", "season"]].drop_duplicates()

train_game_ids = unique_games[unique_games["season"] <= 2023]["game_id"]
test_game_ids = unique_games[unique_games["season"] == 2024]["game_id"]

train_df = modeling_data[modeling_data["game_id"].isin(train_game_ids)].copy()
test_df = modeling_data[modeling_data["game_id"].isin(test_game_ids)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Unique games in train:", train_df["game_id"].nunique())
print("Unique games in test:", test_df["game_id"].nunique())

Train shape: (4916, 98)
Test shape: (570, 98)
Unique games in train: 2458
Unique games in test: 285


In [3]:
# Gathering how many missing counts there is in the data

missing_counts = train_df.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
print(missing_counts)

surface                   82
def_total_yards_roll3     32
adj_def_epa_roll8         32
adj_def_epa_roll3         32
opp_off_epa_season        32
opp_off_epa_roll8         32
opp_off_epa_roll3         32
adj_epa_season            32
adj_epa_roll3             32
adj_epa_roll8             32
opp_def_epa_season        32
opp_def_epa_roll8         32
opp_def_epa_roll3         32
def_total_yards_season    32
def_total_yards_roll8     32
total_yards_season        32
total_yards_roll8         32
total_yards_roll3         32
def_total_epa_season      32
def_total_epa_roll8       32
def_total_epa_roll3       32
total_epa_season          32
total_epa_roll8           32
total_epa_roll3           32
points_against_season     32
points_against_roll8      32
points_against_roll3      32
points_for_season         32
points_for_roll8          32
points_for_roll3          32
adj_def_epa_season        32
referee                    2
dtype: int64


In [ ]:
# fill missing rolling/adjusted stats with 0 (netural)
roll_and_adj_cols = [c for c in train_df.columns if 
                      c.endswith("_roll3") or c.endswith("_roll8") or c.endswith("_season")]

for col in roll_and_adj_cols:
    train_df[col] = train_df[col].fillna(0)
    test_df[col] = test_df[col].fillna(0)

# fill missing categorical columns with a placeholder
train_df["surface"] = train_df["surface"].fillna("unknown")
train_df["referee"] = train_df["referee"].fillna("unknown")
test_df["surface"] = test_df["surface"].fillna("unknown")
test_df["referee"] = test_df["referee"].fillna("unknown")

# confirm no more missing values in either set
print("Remaining missing in train:", train_df.isna().sum().sum())
print("Remaining missing in test:", test_df.isna().sum().sum())

Remaining missing in train: 0
Remaining missing in test: 0


In [5]:
feature_cols = [
    "div_game", "temp", "wind", "rest_days", "opp_rest_days", "is_home",
    "rest_advantage", "short_week", "is_indoor",
    "qb_out", "rb_starter_out", "wr_starter_out",
    "points_for_roll3", "points_for_roll8", "points_for_season",
    "points_against_roll3", "points_against_roll8", "points_against_season",
    "total_epa_roll3", "total_epa_roll8", "total_epa_season",
    "def_total_epa_roll3", "def_total_epa_roll8", "def_total_epa_season",
    "total_yards_roll3", "total_yards_roll8", "total_yards_season",
    "def_total_yards_roll3", "def_total_yards_roll8", "def_total_yards_season",
    "opp_def_epa_roll3", "opp_def_epa_roll8", "opp_def_epa_season",
    "adj_epa_roll3", "adj_epa_roll8", "adj_epa_season",
    "opp_off_epa_roll3", "opp_off_epa_roll8", "opp_off_epa_season",
    "adj_def_epa_roll3", "adj_def_epa_roll8", "adj_def_epa_season",
]

# target: point differential for this game
train_df["point_diff"] = train_df["points_for"] - train_df["points_against"]
test_df["point_diff"] = test_df["points_for"] - test_df["points_against"]

X_train = train_df[feature_cols]
y_train = train_df["point_diff"]

X_test = test_df[feature_cols]
y_test = test_df["point_diff"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train sample:", y_train.head().tolist())

X_train: (4916, 42)
X_test: (570, 42)
y_train sample: [2.0, 4.0, 11.0, 27.0, 6.0]


In [6]:
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression

# baseline 1: dumbest possible guess -- always predict the average point differential from training data
naive_pred = [y_train.mean()] * len(y_test)
naive_mae = mean_absolute_error(y_test, naive_pred)

# baseline 2: simple linear regression using our real features
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
lin_pred = lin_model.predict(X_test)
lin_mae = mean_absolute_error(y_test, lin_pred)

print("Naive baseline MAE:", round(naive_mae, 2))
print("Linear regression MAE:", round(lin_mae, 2))

Naive baseline MAE: 11.3
Linear regression MAE: 10.37


In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print("Naive baseline MAE:", round(naive_mae, 2))
print("Linear regression MAE:", round(lin_mae, 2))
print("XGBoost MAE:", round(xgb_mae, 2))

# Tells me that XGBoost barely beats LR
# Naive baseline MAE: 11.3
# Linear regression MAE: 10.37
# XGBoost MAE: 10.3

Naive baseline MAE: 11.3
Linear regression MAE: 10.37
XGBoost MAE: 10.3


In [ ]:
# figuring out which features model thinks is important

import pandas as pd

importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

print(importances.head(15))

                  feature  importance
13       points_for_roll8    0.083745
37      opp_off_epa_roll8    0.059616
5                 is_home    0.057891
38     opp_off_epa_season    0.053749
20       total_epa_season    0.038131
31      opp_def_epa_roll8    0.031764
16   points_against_roll8    0.030890
40      adj_def_epa_roll8    0.027502
17  points_against_season    0.027226
32     opp_def_epa_season    0.027094
28  def_total_yards_roll8    0.026159
19        total_epa_roll8    0.024894
26     total_yards_season    0.022939
35         adj_epa_season    0.022727
25      total_yards_roll8    0.022307


In [9]:
# isolate test-set rows where the starting QB was out
qb_out_mask = test_df["qb_out"] == 1

qb_out_actual = y_test[qb_out_mask]
qb_out_pred = xgb_pred[qb_out_mask]

qb_out_mae = mean_absolute_error(qb_out_actual, qb_out_pred)
overall_mae = mean_absolute_error(y_test, xgb_pred)

print("Number of QB-out games in test set:", qb_out_mask.sum())
print("MAE on QB-out games:", round(qb_out_mae, 2))
print("MAE overall:", round(overall_mae, 2))

Number of QB-out games in test set: 12
MAE on QB-out games: 18.58
MAE overall: 10.3


In [12]:
# Saving trained data

import joblib

joblib.dump(xgb_model, "../models/xgb_point_diff_v1.pkl")
joblib.dump(feature_cols, "../models/feature_cols_v1.pkl")

print("Model and feature list saved.")

Model and feature list saved.
